# Phase 2 — SFT LoRA puis DPO de Qwen3-1.7B-Base

Fine-tuning supervisé (SFT) de `Qwen/Qwen3-1.7B-Base` sur le dataset anonymisé de la phase 1, avec LoRA, puis alignement par préférences (DPO) du modèle SFT.

- **Prérequis** : GPU, fichiers `data/v4/anonymized/sft_{mediqal_oeq,mediqal_mcqu,mediqal_mcqm,ultramedical}_{train,validation,test}.anonymized.jsonl` produits par `prep_SFT_MediQAl_oeq`, `prep_SFT_MediQAl_mcqu`, `prep_SFT_MediQAl_mcqm` (français) et `prep_SFT_UltraMedical` (anglais), sélectionnés pour le triage (environ 3 000 paires en français, 2 000 en anglais). **Une ligne = un cas clinique** : le cas, puis ses questions dans l'ordre, chacune suivie de sa réponse (questions ouvertes, ou QCM avec propositions A à E et consigne). La perte porte sur **toutes les réponses de l'assistant** (`assistant_only_loss`), chacune avec l'historique qui la précède.
- **Prérequis DPO** : fichiers `data/v4/anonymized/dpo_{mediqal,ultramedical}_{train,validation,test}.anonymized.jsonl` produits par `prep_DPO_MediQAl` (français : QCM, `rejected` = corrigé avec une erreur construite) et `prep_DPO_UltraMedical` (anglais : paires vérifiées par le corrigé ou par un écart de note net, test revu par des experts).
- **Étapes** : données → modèle → LoRA → entraînement SFT → validation intermédiaire → DPO → validation DPO → modèle final fusionné.
- **Suivi** : MLflow sur DagsHub (`MLFLOW_BACKEND = "dagshub"`), consultable de partout et quelle que soit la machine d'entraînement. En local (`"local"`) : `mlflow ui --backend-store-uri sqlite:///mlflow.db` depuis la racine du projet.

In [1]:
# Installation (Colab ou machine GPU) : décommenter si besoin
# %pip install -q torch transformers peft trl datasets accelerate mlflow dagshub pandas

## 1. Configuration

In [1]:
import functools
import gc
import json
import math
import re
from collections import Counter
from pathlib import Path

import mlflow
import pandas as pd
import torch
from datasets import DatasetDict, concatenate_datasets, load_dataset
from IPython.display import display
from peft import LoraConfig, PeftModel
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer, SFTConfig, SFTTrainer
from trl.chat_template_utils import is_chat_template_stop_token_trained

from projet_14.mediqal import LETTERS, OPTIONS_HEADER, answer_letters

MODEL_NAME = "Qwen/Qwen3-1.7B-Base"
# Données SFT de la phase 1, anonymisées et sélectionnées pour le triage (~3 000 paires FR, 2 000 EN) : fichier -> langue.
# Une ligne = un cas clinique (conversation multi-tours) ; en anglais, une question
SFT_DATA_DIR = Path("../data/v4/anonymized")
SFT_DATASETS = {
    "sft_mediqal_oeq": "fr",    # questions ouvertes, réponses rédigées, avec cas clinique
    "sft_mediqal_mcqu": "fr",   # QCM à réponse unique, avec cas clinique
    "sft_mediqal_mcqm": "fr",   # QCM à réponses multiples (liste), avec cas clinique
    "sft_ultramedical": "en",   # vignettes cliniques (MedQA…) et patients (ChatDoctor), réponses rédigées en anglais
}

# Suivi MLflow : "dagshub" (serveur distant) ou "local" (fichier ../mlflow.db)
MLFLOW_BACKEND = "dagshub"
DAGSHUB_REPO_OWNER = "obiyohan"
DAGSHUB_REPO_NAME = "my-ml-mlflow"

# Petit run pour valider la pipeline avant l'entraînement complet
SMALL_RUN = False  # True : run rapide (quelques lignes par source, dont les plus longues), False : run complet

SEED = 42
# Version des données de la phase 1 : dans le nom des runs, pour ne pas écraser les checkpoints d'une version à l'autre
# v1 : propositions absentes du prompt ; v2 : propositions dans le prompt, répartition par cas clinique,
#      DPO anglais équilibré sur la longueur ; v3 : consigne « une seule / plusieurs propositions exactes » ;
# v4 : une ligne par cas clinique (questions dans l'ordre du récit), questions ouvertes ajoutées, perte sur toutes les réponses
DATA_VERSION = "v4"
RUN_NAME = f"sft-lora-r16-{DATA_VERSION}-seed{SEED}" + ("-small" if SMALL_RUN else "")
OUTPUT_DIR = Path("../outputs") / RUN_NAME  # checkpoints et adaptateur final

set_seed(SEED)
if torch.cuda.is_available():
    device, bf16 = "cuda", torch.cuda.is_bf16_supported()  # bf16 absent sur T4 : fp16 à la place
elif torch.xpu.is_available():
    device, bf16 = "xpu", torch.xpu.is_bf16_supported()  # GPU Intel Arc
else:
    device, bf16 = "cpu", False
dtype = torch.bfloat16 if bf16 else torch.float16 if device != "cpu" else torch.float32
print(f"device={device}, dtype={dtype}")

if device == "xpu":
    # GPU Intel sous Windows : deux contournements
    # 1. Pas de compilateur C pour Triton : TRL calcule la loss en PyTorch pur
    import trl.kernels.chunked_logprob as chunked_logprob
    chunked_logprob._TRITON_AVAILABLE = False
    # 2. La norme des gradients en mode « foreach » fait planter le GPU : calcul tenseur par tenseur
    torch.nn.utils.clip_grad_norm_ = functools.partial(torch.nn.utils.clip_grad.clip_grad_norm_, foreach=False)

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device=cuda, dtype=torch.bfloat16


## 2. Données : train / validation / test

In [3]:
# Format conversationnel (colonne `messages`) : une ligne = un cas clinique et toutes ses questions.
# Avec assistant_only_loss (section 5), la perte porte sur toutes les réponses de l'assistant, pas sur le cas ni les questions.
# Chaque source garde son découpage train / validation / test (tiré du cas clinique) ; les sources sont ensuite mélangées
parts = {split: [] for split in ("train", "validation", "test")}
for stem, lang in SFT_DATASETS.items():
    files = {split: str(SFT_DATA_DIR / f"{stem}_{split}.anonymized.jsonl") for split in parts}
    ds = load_dataset("json", data_files=files).select_columns(["messages", "n_questions"])
    for split, part in ds.items():
        parts[split].append(part.add_column("source", [stem] * len(part)).add_column("lang", [lang] * len(part)))

splits = DatasetDict({split: concatenate_datasets(p).shuffle(seed=SEED) for split, p in parts.items()})

# Petit run : quelques lignes de chaque source, pour que chaque format et chaque évaluation soient exercés,
# plus les lignes les plus longues du train, pour tester la mémoire GPU dans le pire cas
if SMALL_RUN:
    SMALL_SIZES = {"train": 12, "validation": 4, "test": 6}  # lignes par source
    SMALL_LONGEST = 4                                        # + lignes les plus longues du train
    for name, size in SMALL_SIZES.items():
        ds = splits[name]
        keep = [i for stem in SFT_DATASETS for i in [j for j, s in enumerate(ds["source"]) if s == stem][:size]]
        if name == "train":
            counter = AutoTokenizer.from_pretrained(MODEL_NAME)  # longueur en tokens (en caractères, l'anglais serait favorisé)
            length = [len(counter("\n".join(m["content"] for m in messages))["input_ids"]) for messages in ds["messages"]]
            keep += sorted(range(len(ds)), key=lambda i: length[i], reverse=True)[:SMALL_LONGEST]
        splits[name] = ds.select(sorted(set(keep)))

# Lignes (cas) et questions (réponses apprises) par source et par split
pd.concat({
    "lignes": pd.DataFrame({name: pd.Series(ds["source"]).value_counts() for name, ds in splits.items()}),
    "questions": pd.DataFrame({name: pd.DataFrame({"source": ds["source"], "n": ds["n_questions"]}).groupby("source")["n"].sum()
                               for name, ds in splits.items()}),
}, axis=1).fillna(0).astype(int)


lignes                 questions                
                  train validation test     train validation test
sft_mediqal_mcqm    185         27   51       490         70  140
sft_mediqal_mcqu    253         37   63       840        120  240
sft_mediqal_oeq     248         37   84       770        109  219
sft_ultramedical   1400        200  400      1400        200  400

## 3. Tokenizer et modèle

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Gabarit ChatML, avec une réponse de l'assistant terminée par <|endoftext|> (eos_token) au lieu de <|im_end|>.
# Dans Qwen3-Base, <|im_start|> et <|im_end|> n'ont jamais été entraînés (embeddings à leur valeur d'initialisation)
# et LoRA ne modifie pas les embeddings : le modèle ne peut pas apprendre à produire <|im_end|>, donc à s'arrêter.
# <|endoftext|> a été appris au pré-entraînement : le modèle sait le prédire, et vLLM s'arrête dessus par défaut.
# {% generation %} ... {% endgeneration %} : délimite les tokens appris (assistant_only_loss). Ils n'apparaissent pas dans
# le texte rendu : le prompt vu par vLLM est identique. <|endoftext|> est à l'intérieur : le modèle apprend à s'arrêter.
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' }}"
    "{% if message['role'] == 'assistant' %}"
    "{% generation %}{{ message['content'] + eos_token }}{% endgeneration %}"
    "{% else %}{{ message['content'] + '<|im_end|>\n' }}{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)
assert is_chat_template_stop_token_trained(tokenizer), "<|endoftext|> doit être dans le masque des réponses"

# Une ligne à plusieurs questions, et les tokens appris (masque des réponses de l'assistant)
example = next(e for e in splits["train"] if e["n_questions"] >= 3)
print(tokenizer.apply_chat_template(example["messages"], tokenize=False))
encoded = tokenizer.apply_chat_template(example["messages"], tokenize=True, return_dict=True, return_assistant_tokens_mask=True)
learned = [token for token, mask in zip(encoded["input_ids"], encoded["assistant_masks"]) if mask]
print(f"=== {len(learned)} tokens appris sur {len(encoded['input_ids'])} :\n{tokenizer.decode(learned)}")


<|im_start|>system
Tu es un assistant médical expert. Réponds de façon précise et concise.<|im_end|>
<|im_start|>user
Cas clinique : Une femme de 55 ans vient consulter pour un ictère apparu progressivement, sans fièvre, ni douleur abdominale. Dans ses antécédents, on note une hypertension artérielle traitée depuis deux ans par de l'Aldomet® (alphaméthyl-dopa), et pour une arthrose cervico-dorsale du Dupéran (clométacine). Elle est en outre constipée chronique et prend depuis plusieurs années des laxatifs (Bisacodyl, Séné, Dioctylsulfosuccinate). Elle consomme en moyenne 20 g d'alcool par jour. A l'examen clinique, on note une ascite modérée, des oedèmes des membres inférieurs ; elle se plaint d'une asthénie avec somnolence diurne et parfois désorientation. Biologiquement, il existe un bloc bêta-gamma à l'électrophorèse des protides, une vitesse de sédimentation à 35 mm à la première heure, des triglycérides à 1, 2 mmol/l ; un fer sérique normal, une ferritinémie à une fois et demie la

In [5]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device)
print(f"{model.num_parameters() / 1e9:.2f} B paramètres")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1249.61it/s]


1.72 B paramètres


## 4. LoRA

In [6]:
# Seules de petites matrices de rang r sont entraînées : le modèle de base reste figé
lora_config = LoraConfig(
    r=16,                         # rang des matrices d'adaptation
    lora_alpha=32,                # facteur d'échelle (souvent 2 × r)
    lora_dropout=0.05,
    target_modules="all-linear",  # couches d'attention et MLP
    task_type="CAUSAL_LM",
)

## 5. Entraînement SFT

In [7]:
# Suivi MLflow : un run par entraînement, hyperparamètres et métriques enregistrés
if MLFLOW_BACKEND == "dagshub":
    import dagshub

    # Jeton lu dans la variable d'environnement DAGSHUB_USER_TOKEN,
    # sinon connexion via le navigateur au premier lancement (jeton mémorisé ensuite)
    dagshub.init(repo_owner=DAGSHUB_REPO_OWNER, repo_name=DAGSHUB_REPO_NAME, mlflow=True)
else:
    mlflow.set_tracking_uri("sqlite:///../mlflow.db")
print(f"Suivi MLflow : {mlflow.get_tracking_uri()}")

mlflow.set_experiment("qwen3-1.7b-sft-lora")
mlflow.start_run(run_name=RUN_NAME)
mlflow.log_params({"model": MODEL_NAME, "data_version": DATA_VERSION, "assistant_only_loss": True, "small_run": SMALL_RUN, "device": device, "lora_r": lora_config.r, "lora_alpha": lora_config.lora_alpha, "lora_dropout": lora_config.lora_dropout})

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    run_name=RUN_NAME,
    seed=SEED,
    # Optimisation
    # Une époque : au run v2 (2 époques), eval_loss était au plus bas en fin d'époque 1 puis remontait légèrement
    num_train_epochs=1,
    max_steps=30 if SMALL_RUN else -1,
    # v4 : lignes plus longues (un cas entier, jusqu'à ~1 460 tokens) : 2 lignes par lot, même batch effectif
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,  # batch effectif = 16 lignes (cas cliniques)
    per_device_eval_batch_size=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,  # 3 % des étapes en montée progressive du learning rate
    max_length=1536,  # aucune ligne v4 n'est coupée (max ~1 460 tokens) : toutes les réponses sont apprises
    assistant_only_loss=True,  # perte sur toutes les réponses de l'assistant, pas sur le cas ni les questions
    gradient_checkpointing=True,  # moins de mémoire GPU, un peu plus lent
    bf16=bf16,
    fp16=dtype == torch.float16,
    # Évaluation et checkpoints
    eval_strategy="steps",
    # v4 : ~2 100 lignes en train, soit ~130 étapes par époque : une évaluation et un checkpoint toutes les 25 étapes
    eval_steps=10 if SMALL_RUN else 25,
    save_steps=10 if SMALL_RUN else 25,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    # Logs
    logging_steps=5,
    report_to="mlflow",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=splits["train"],
    eval_dataset=splits["validation"],
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()

Accessing as ObiYohan

Initialized MLflow to track repo "obiyohan/my-ml-mlflow"

Repository obiyohan/my-ml-mlflow initialized!

Suivi MLflow : https://dagshub.com/obiyohan/my-ml-mlflow.mlflow


Truncating train dataset: 100%|██████████| 2086/2086 [00:00<00:00, 3195.56 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 2086/2086 [00:00<00:00, 16935.14 examples/s]
Dropping fully masked examples from eval dataset: 100%|██████████| 301/301 [00:00<00:00, 14060.11 examples/s]

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


In [8]:
# Référence avant entraînement (adaptateur LoRA initialisé à zéro = modèle de base)
val_before = trainer.evaluate(metric_key_prefix="val_before")
val_before

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)


Training Loss,Validation Loss,Step,Before Loss,Entropy,Mean Token Accuracy,Num Tokens
No log,No log,0,1.089840,1.066209,0.708230,0.000000


{'val_before_loss': 1.0898404121398926,
 'eval_entropy': 1.0662088409850472,
 'eval_mean_token_accuracy': 0.7082303629109734,
 'eval_num_tokens': 0.0}

In [9]:
# Reprise après interruption : trainer.train(resume_from_checkpoint=True)
train_result = trainer.train()
train_result.metrics

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
25,1.109204,0.995239,1.005677,0.727889,256637.000000
50,0.925803,0.982358,0.982665,0.730872,513749.000000
75,1.066154,0.975767,0.979746,0.732138,778984.000000
100,1.001814,0.972800,0.954027,0.733226,1045192.000000
125,1.050283,0.972025,0.961579,0.732401,1308884.000000
131,0.919917,0.971668,0.961286,0.733007,1364756.000000


{'train_runtime': 1129.2263,
 'train_samples_per_second': 1.847,
 'train_steps_per_second': 0.116,
 'total_flos': 1.3690360926683136e+16,
 'train_loss': 0.9870406789634064,
 'epoch': 1.0}

In [10]:
# Sauvegarde de l'adaptateur LoRA (meilleur checkpoint) et de la configuration du run
FINAL_DIR = OUTPUT_DIR / "final"
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))

run_config = {
    "model": MODEL_NAME,
    "seed": SEED,
    "small_run": SMALL_RUN,
    "lora": {"r": lora_config.r, "alpha": lora_config.lora_alpha, "dropout": lora_config.lora_dropout},
    "data_version": DATA_VERSION,
    "data": {name: {"lignes": len(ds), "questions": sum(ds["n_questions"])} for name, ds in splits.items()},
    "training_args": training_args.to_dict(),
}
(FINAL_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2, default=str), encoding="utf-8")

4743

## 6. Validation intermédiaire

1. **Loss, perplexité et token accuracy** sur validation (avant / après SFT) et test.
2. **Qualité des réponses générées** sur toutes les questions de test (QCM `mcqu` et `mcqm`, dans l'ordre des fichiers ; chaque question d'un cas avec l'historique de référence qui la précède), modèle de base vs modèle SFT, sur les **lettres** données dans la réponse. Les mêmes questions servent en 7.4. Génération par lots pour bien charger le GPU.
   - `accuracy` : exactement les lettres attendues ;
   - `option_accuracy` : exactitude par proposition, part des cinq propositions correctement cochées ou laissées. Utile en `mcqm` ; en `mcqu`, une réponse vide obtient déjà 0,8 : y lire l'accuracy ;
   - `f1` : crédit partiel pour les réponses multiples. Il favorise les réponses qui cochent trop : en `mcqm`, tout cocher donne un F1 d'environ 0,7 ;
   - `extra_letters` : lettres données en trop (positif) ou en moins (négatif) par rapport au corrigé, en moyenne ;
   - `format` : part des réponses au format appris, les propositions retenues recopiées telles quelles, sans explication ni texte en plus (modèles seulement).

   Deux **réponses naïves**, identiques pour toutes les questions, servent de référence : `frequente` (la réponse la plus fréquente du train) et `tout_cocher` (A à E). Un modèle utile doit les battre, sur l'accuracy et l'exactitude par proposition.

La **latence** n'est pas mesurée ici : avec l'adaptateur LoRA non fusionné et Transformers, elle ne serait pas représentative. Elle sera mesurée en phase 3, sur le modèle final fusionné servi par vLLM.

In [11]:
val_after = trainer.evaluate(metric_key_prefix="val_after")
test_metrics = trainer.evaluate(splits["test"], metric_key_prefix="test")

# token_accuracy : part des tokens de la réponse attendue que le modèle prédit correctement
# (TRL la nomme « eval_mean_token_accuracy » quel que soit le préfixe : on la renomme ici)
evaluations = {"val_before": val_before, "val_after": val_after, "test": test_metrics}
summary = pd.DataFrame({
    name: {"loss": m[f"{name}_loss"], "token_accuracy": m["eval_mean_token_accuracy"]}
    for name, m in evaluations.items()
}).T
summary["perplexity"] = summary["loss"].map(math.exp)

# Même mesure sur le test, source par source : chaque format doit être appris (oeq : réponses rédigées, évaluées ici seulement)
per_source = {}
for source in SFT_DATASETS:
    subset = splits["test"].filter(lambda example: example["source"] == source)
    if len(subset):
        m = trainer.evaluate(subset, metric_key_prefix=f"test_{source}")
        per_source[f"test_{source}"] = {"loss": m[f"test_{source}_loss"], "token_accuracy": m["eval_mean_token_accuracy"]}
per_source_summary = pd.DataFrame(per_source).T
per_source_summary["perplexity"] = per_source_summary["loss"].map(math.exp)
pd.concat([summary, per_source_summary])

Training Loss,Validation Loss,Step,After Loss,Entropy,Mean Token Accuracy,Num Tokens
0.919917,No log,131,0.971668,0.961286,0.733007,1364756.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 598/598 [00:00<00:00, 14905.68 examples/s]


Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
0.919917,0.973444,131,0.971076,0.731432,1364756.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 84/84 [00:00<00:00, 8187.62 examples/s]


Training Loss,Validation Loss,Step,Sft Mediqal Oeq Loss,Entropy,Mean Token Accuracy,Num Tokens
0.919917,No log,131,2.144418,2.160700,0.554279,1364756.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 63/63 [00:00<00:00, 6171.41 examples/s]


... (2 outputs left out)

Training Loss,Validation Loss,Step,Sft Mediqal Mcqm Loss,Entropy,Mean Token Accuracy,Num Tokens
0.919917,No log,131,0.090485,0.091707,0.962029,1364756.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 400/400 [00:00<00:00, 15991.40 examples/s]


Training Loss,Validation Loss,Step,Sft Ultramedical Loss,Entropy,Mean Token Accuracy,Num Tokens
0.919917,No log,131,0.852393,0.845845,0.742027,1364756.000000


,loss,token_accuracy,perplexity
val_before,1.089840,0.708230,2.973799
val_after,0.971668,0.733007,2.642347
test,0.973444,0.731432,2.647046
test_sft_mediqal_oeq,2.144418,0.554279,8.537073
test_sft_mediqal_mcqu,0.126026,0.952243,1.134311
test_sft_mediqal_mcqm,0.090485,0.962029,1.094705
test_sft_ultramedical,0.852393,0.742027,2.345252


In [2]:
# Arrêt de la génération : <|im_end|> ou eos. Le tokenizer est lu à l'appel : cette cellule peut être exécutée avant
# la section 7 sans les sections 2 à 5 (tokenizer chargé en 7.1)
# Réponses mcqu : « C. texte de la proposition ». La lettre vient en premier : une réponse coupée reste évaluable.
# Le modèle de base, qui ne s'arrête pas toujours, est ainsi coupé plus tôt.
MAX_NEW_TOKENS = 64


def render(prompt):
    return tokenizer.apply_chat_template(prompt, add_generation_prompt=True, tokenize=False)


@torch.no_grad()
def generate_batch(prompts, batch_size=16, max_new_tokens=MAX_NEW_TOKENS, desc=None, model=None):
    # Génération par lots (GPU bien chargé) : renvoie les réponses, dans l'ordre des prompts
    model = model or trainer.model  # par défaut, le modèle du SFT
    texts = [render(prompt) for prompt in prompts]
    # Prompts triés par longueur : moins de padding dans chaque lot, les plus longs d'abord
    # (un manque de mémoire GPU apparaît dès le premier lot)
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]), reverse=True)
    answers = [None] * len(texts)
    padding_side, tokenizer.padding_side = tokenizer.padding_side, "left"  # padding à gauche : la génération suit chaque prompt
    try:
        for start in tqdm(range(0, len(order), batch_size), desc=desc, unit="lot"):
            batch = order[start:start + batch_size]
            inputs = tokenizer([texts[i] for i in batch], return_tensors="pt", padding=True, add_special_tokens=False).to(device)
            output = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                eos_token_id=[tokenizer.convert_tokens_to_ids("<|im_end|>"), tokenizer.eos_token_id],
                pad_token_id=tokenizer.pad_token_id,
            )
            decoded = tokenizer.batch_decode(output[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            for i, answer in zip(batch, decoded):
                answers[i] = answer.strip()
    finally:
        tokenizer.padding_side = padding_side
    return answers


def question_examples(dataset):
    # Une ligne (cas) -> une question par réponse de l'assistant : prompt = conversation jusqu'à elle, référence = sa réponse
    prompts, references = [], []
    for messages in dataset["messages"]:
        for k, message in enumerate(messages):
            if message["role"] == "assistant":
                prompts.append(messages[:k])
                references.append(message["content"])
    return prompts, references


def test_questions(source, n=None):
    # Questions de test d'une source SFT française, dans l'ordre du fichier : les mêmes en section 6 et en 7.4.
    # n : les n premières (petit run) ; None : toutes
    data = load_dataset("json", data_files={"test": str(SFT_DATA_DIR / f"{source}_test.anonymized.jsonl")})["test"]
    prompts, references = question_examples(data)
    return prompts[:n], references[:n]


QUALITY_METRICS = ["accuracy", "option_accuracy", "f1", "extra_letters"]
FORMAT_METRIC = "format"  # modèles seulement : les réponses naïves n'ont que des lettres
NAIVE_ANSWERS = ["frequente", "tout_cocher"]


def letter_scores(predicted: frozenset, expected: frozenset) -> dict:
    # Mesures sur les lettres d'une réponse (ensembles de lettres A à E)
    common = len(predicted & expected)
    return {
        "accuracy": float(predicted == expected),  # exactement les lettres attendues
        # Exactitude par proposition : part des 5 propositions correctement cochées ou laissées
        "option_accuracy": sum((letter in predicted) == (letter in expected) for letter in LETTERS) / len(LETTERS),
        # F1 : crédit partiel ; favorise les réponses qui cochent trop (tout cocher : ~0,7 en mcqm)
        "f1": 2 * common / (len(predicted) + len(expected)) if common else 0.0,
        "extra_letters": len(predicted) - len(expected),  # > 0 : coche trop ; < 0 : pas assez
    }


def letters_text(letters) -> str:
    # Réponse réduite à ses lettres (« A.\nC. »), lisible par answer_letters
    return "\n".join(f"{letter}." for letter in sorted(letters))


def naive_answers(train_answers: list[str]) -> dict:
    # Réponses naïves, identiques pour toutes les questions : un modèle utile doit les battre
    most_common = Counter(answer_letters(answer) for answer in train_answers).most_common(1)[0][0]
    return {"frequente": letters_text(most_common), "tout_cocher": letters_text(LETTERS)}


def answer_format_ok(answer: str, prompt: list[dict]) -> bool:
    # Format appris : les propositions retenues recopiées telles quelles depuis le prompt (« C. texte », ou une liste
    # « - C. texte »), sans explication ni texte en plus. Vrai pour 100 % des réponses de référence (v4).
    block = prompt[-1]["content"].split(OPTIONS_HEADER, 1)[-1].strip().split("\n\n")[0]
    parts = re.split(r"(?m)^([A-E])\. ", block)
    options = dict(zip(parts[1::2], parts[2::2]))
    letters = sorted(answer_letters(answer))

    def normalize(text):
        return " ".join(re.sub(r"(?m)^\s*- ", "", text).split())

    return (bool(letters) and all(letter in options for letter in letters)
            and normalize(answer) == normalize("\n".join(f"{letter}. {options[letter]}" for letter in letters)))


def add_scores(df: pd.DataFrame, columns: list[str], prompts: list[list[dict]]) -> pd.DataFrame:
    # Ajoute <mesure>_<colonne> pour chaque colonne de réponses, comparée à df["reference"] ;
    # format_<colonne> pour les modèles (prompts : ceux de chaque ligne de df, dans l'ordre)
    expected = [answer_letters(reference) for reference in df["reference"]]
    for column in columns:
        scores = pd.DataFrame(
            [letter_scores(answer_letters(answer), e) for answer, e in zip(df[column], expected)], index=df.index
        )
        for metric in QUALITY_METRICS:
            df[f"{metric}_{column}"] = scores[metric]
        if column not in NAIVE_ANSWERS:
            df[f"{FORMAT_METRIC}_{column}"] = [float(answer_format_ok(a, p)) for a, p in zip(df[column], prompts)]
    return df


def score_table(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    # Moyennes : une ligne par mesure, une colonne par modèle ou réponse naïve (format : NaN pour les réponses naïves)
    return pd.DataFrame({
        column: {metric: df[f"{metric}_{column}"].mean() if f"{metric}_{column}" in df else float("nan")
                 for metric in [*QUALITY_METRICS, FORMAT_METRIC]}
        for column in columns
    })


def mcnemar_p(only_a: int, only_b: int) -> float:
    # Test de McNemar exact (binomial bilatéral) : seules comptent les questions où un seul des deux modèles a raison
    n, k = only_a + only_b, min(only_a, only_b)
    return 1.0 if n == 0 else min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)


def paired_comparison(df: pd.DataFrame, a: str, b: str) -> dict:
    # Questions où seul a (ou seul b) donne exactement les bonnes lettres, et p-value de McNemar
    only_a = int(((df[f"accuracy_{a}"] == 1) & (df[f"accuracy_{b}"] == 0)).sum())
    only_b = int(((df[f"accuracy_{b}"] == 1) & (df[f"accuracy_{a}"] == 0)).sum())
    return {"questions": len(df), f"only_{a}": only_a, f"only_{b}": only_b, "p_value": mcnemar_p(only_a, only_b)}


def letters_by_expected(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    # Lettres données en moyenne selon le nombre de bonnes réponses attendues (mcqm) : un bon modèle suit ce nombre
    expected = df["reference"].map(lambda reference: len(answer_letters(reference))).rename("attendues")
    table = pd.DataFrame({column: df[column].map(lambda answer: len(answer_letters(answer))).groupby(expected).mean()
                          for column in columns})
    return table.assign(questions=expected.value_counts().sort_index())

In [13]:
N_EVAL = 20 if SMALL_RUN else None   # questions de test par source (None : toutes, 240 mcqu et 140 mcqm)
BATCH_SIZE = 16                      # à baisser en cas de manque de mémoire GPU (OutOfMemoryError)

# Sources évaluées par génération, avec leur nombre maximal de tokens générés.
# Les réponses rédigées (oeq, anglais) ne se comparent pas mot à mot à une référence :
# elles sont évaluées par la loss et la perplexité (cellule précédente).
GENERATION_SOURCES = {"sft_mediqal_mcqu": MAX_NEW_TOKENS, "sft_mediqal_mcqm": 192}
ANSWER_COLUMNS = ["base", "sft", *NAIVE_ANSWERS]  # modèles, puis réponses naïves de référence

trainer.model.eval()
trainer.model.config.use_cache = True  # accélère la génération

# Génération par lots, modèle de base puis modèle SFT, source par source
source_generations, generation_prompts = [], []
for source, max_new_tokens in GENERATION_SOURCES.items():
    # Chaque question d'un cas, avec l'historique de référence (questions et bonnes réponses précédentes) ;
    # dans l'ordre du fichier : mêmes questions qu'en 7.4
    prompts, references = test_questions(source, N_EVAL)
    with trainer.model.disable_adapter():  # modèle de base, sans LoRA
        base_answers = generate_batch(prompts, BATCH_SIZE, max_new_tokens, desc=f"Base {source}")
    sft_answers = generate_batch(prompts, BATCH_SIZE, max_new_tokens, desc=f"SFT {source}")
    train_answers = question_examples(splits["train"].filter(lambda example: example["source"] == source))[1]
    source_generations.append(pd.DataFrame({
        "source": source,
        "question": [prompt[-1]["content"][-200:] for prompt in prompts],
        "reference": references,
        "base": base_answers,
        "sft": sft_answers,
        **naive_answers(train_answers),  # mêmes réponses pour toutes les questions de la source
    }))
    generation_prompts += prompts
generations = add_scores(pd.concat(source_generations, ignore_index=True), ANSWER_COLUMNS, generation_prompts)
generations.to_csv(FINAL_DIR / "test_generations.csv", index=False)

# Questions où un seul des deux modèles a la bonne réponse (McNemar), par source
display(pd.DataFrame({source: paired_comparison(group, "sft", "base") for source, group in generations.groupby("source")}).T)
# Moyennes sur toutes les sources, puis source par source
comparison = score_table(generations, ANSWER_COLUMNS)
quality_by_source = pd.concat({source: score_table(group, ANSWER_COLUMNS) for source, group in generations.groupby("source")})
display(quality_by_source)
comparison


Filter:   0%|          | 0/598 [00:00<?, ? examples/s]

Filter: 100%|██████████| 2086/2086 [00:00<00:00, 65240.84 examples/s]


base       sft  frequente  tout_cocher
sft_mediqal_mcqm accuracy         0.042857  0.071429   0.028571     0.042857
                 option_accuracy  0.550000  0.615714   0.541429     0.532857
                 f1               0.522812  0.660573   0.586293     0.681491
                 extra_letters    0.821429  0.678571   0.335714     2.335714
sft_mediqal_mcqu accuracy         0.170000  0.465000   0.230000     0.000000
                 option_accuracy  0.651000  0.786000   0.692000     0.200000
                 f1               0.287667  0.465000   0.230000     0.333333
                 extra_letters    0.615000  0.000000   0.000000     4.000000

,base,sft,frequente,tout_cocher
accuracy,0.117647,0.302941,0.147059,0.017647
option_accuracy,0.609412,0.715882,0.630000,0.337059
f1,0.384491,0.545530,0.376709,0.476692
extra_letters,0.700000,0.279412,0.138235,3.314706


In [14]:
# Quelques exemples à relire
pd.set_option("display.max_colwidth", 200)
generations.groupby("source")[["source", "question", "reference", "base", "sft"]].head(3)

,source,question,reference,base,sft
0,sft_mediqal_mcqu,ale\nC. L'adaptation du traitement de l'hypertension artérielle\nD. La mise en place d'une perfusion de mannitol a 20%\nE. La protection des voies aériennes supérieures\n\nUne seule proposition es...,D. La mise en place d'une perfusion de mannitol a 20%,La réponse fausse est : **A. L'administration de diazépam par voie générale**.\n\nExplication : \nLa pré-eclampsie est une condition hypertensive obstétricale caractérisée par une hypertension ar...,D. La mise en place d'une perfusion de mannitol a 20%
1,sft_mediqal_mcqu,Une réanimation métabolique seule\nD. Une intubation associée à une ventilation et une réanimation métabolique\nE. Une intubation associée à une ventilation uniquement\n\nUne seule proposition est...,A. Une désobstruction associée à de l'oxygène nasal,D. Une intubation associée à une ventilation et une réanimation métabolique\n.waitKey\n.waitKey\n.waitKey\n.waitKey\n.waitKey\n.waitKey\n.waitKey\n.waitKey\n.waitKey\n.waitKey\n.waitKey\n.waitKey\...,D. Une intubation associée à une ventilation et une réanimation métabolique
2,sft_mediqal_mcqu,Le DIU (dispositif intra-utérin)\nB. La pilule micro-progestative en continu\nC. Les macro-progestatifs\nD. La pilule minidosée\nE. Les propositions A et C sont exactes\n\nUne seule proposition e...,A. Le DIU (dispositif intra-utérin),A. Le DIU (dispositif intra-utérin)\nB. La pilule micro-progestative en continu\nC. Les macro-progestatifs\nD. La pilule minidosée\nE. Les propositions A et C sont exactes\n\nUne seule proposition...,E. Les propositions A et C sont exactes
200,sft_mediqal_mcqm,. Ce taux est compatible avec le diagnostic de méningite virale\nE. Ce taux est compatible avec le diagnostic de méningite à Listeria monocytogénès\n\nPlusieurs propositions sont exactes (au moins...,- C. Ce taux évoque très fortement la présence d'une bactérie dans le LCR\n- E. Ce taux est compatible avec le diagnostic de méningite à Listeria monocytogénès,Tu es un assistant médical expert. Réponds de façon précise et concise.��态\n��态：Tu es un assistant médical expert. Réponds de façon précise et concise.��态\n��态：Tu es un assistant médical expert. R...,- B. Il s'agit d'un taux incompatible avec le diagnostic de méningite tuberculeuse\n- C. Ce taux évoque très fortement la présence d'une bactérie dans le LCR
201,sft_mediqal_mcqm,. Myélogramme\nC. Gastroscopie avec biopsie\nD. Dosages de vitamine B12 et de folates dans le sang\nE. Recherche d'anticorps anti-facteur intrinsèque\n\nPlusieurs propositions sont exactes (au moi...,- B. Myélogramme\n- D. Dosages de vitamine B12 et de folates dans le sang,Tu es un assistant médical expert. Réponds de façon précise et concise.��态\n��态：Tu es un assistant médical expert. Réponds de façon précise et concise.��态\n状态：Tu es un assistant médical expert. Ré...,- A. Tubage gastrique\n- B. Myélogramme\n- C. Gastroscopie avec biopsie\n- D. Dosages de vitamine B12 et de folates dans le sang
202,sft_mediqal_mcqm,B. Trouble d'absorption de la vitamine B12\nC. Carence d'apport en folates\nD. Trouble d'absorption des folates\nE. Trouble d'utilisation des folates\n\nPlusieurs propositions sont exactes (au moi...,- C. Carence d'apport en folates\n- D. Trouble d'absorption des folates,- A. Carence d'apport en vitamine B12\n- C. Carence d'apport en folates\n- D. Trouble d'absorption des folates\n- E. Trouble d'utilisation des folates\n\nQuestion : Quel est le principal facteur c...,- A. Carence d'apport en vitamine B12\n- C. Carence d'apport en folates


In [15]:
# Métriques finales dans MLflow (les loss sont déjà envoyées par le Trainer), puis clôture du run
mlflow.log_metrics({f"{name}_token_accuracy": row["token_accuracy"] for name, row in summary.iterrows()})
mlflow.log_metrics({f"{name}_perplexity": row["perplexity"] for name, row in summary.iterrows()})
mlflow.log_metrics({
    f"test_{metric}_{model}": value
    for metric, row in comparison.iterrows()
    for model, value in row.items()
    if pd.notna(value)  # format : modèles seulement
})
# Par source : loss, token accuracy et perplexité du test, mesures des réponses générées et des réponses naïves
mlflow.log_metrics({
    f"{name}_{metric}": value for name, row in per_source_summary.iterrows() for metric, value in row.items()
})
mlflow.log_metrics({
    f"test_{source}_{metric}_{column}": value
    for (source, metric), row in quality_by_source.iterrows()
    for column, value in row.items()
    if pd.notna(value)
})
mlflow.end_run()

🏃 View run sft-lora-r16-v4-seed42 at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/2/runs/33e7d6cde9374059b82b769bf40c2398
🧪 View experiment at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/2


## Lecture des résultats

| Métrique MLflow | Définition | Attendu |
|---|---|---|
| `loss`, `eval_loss` | courbes de loss d'entraînement et de validation | baissent ensemble ; si `eval_loss` remonte : surapprentissage |
| `val_before_loss`, `val_after_loss`, `test_loss` | loss avant / après SFT et sur le test | après < avant ; test proche de val |
| `*_token_accuracy` | part des tokens de la réponse correctement prédits | augmente après SFT |
| `*_perplexity` | exp(loss) | baisse après SFT |
| `test_sft_<source>_loss`, `_token_accuracy`, `_perplexity` | les mêmes mesures sur le test, source par source | chaque source s'améliore, y compris les questions ouvertes (`oeq`) |
| `test_accuracy_*` | réponse générée avec exactement les lettres attendues (sources françaises) | sft > base, sft > `frequente` et `tout_cocher` |
| `test_option_accuracy_*` | exactitude par proposition : part des 5 propositions correctement cochées ou laissées | sft > `frequente` et `tout_cocher` en `mcqm` ; en `mcqu`, une réponse vide obtient 0,8 : lire l'accuracy |
| `test_f1_*` | F1 sur les lettres (crédit partiel pour les réponses multiples) | sft > base ; à lire avec prudence, `tout_cocher` obtient ~0,7 en `mcqm` |
| `test_extra_letters_*` | lettres en trop (> 0) ou en moins (< 0) par rapport au corrigé | proche de 0 |
| `test_format_*` | part des réponses au format appris (propositions recopiées, sans texte en plus) | sft proche de 1 |
| `test_sft_<source>_<mesure>_*` | les mêmes mesures, source par source | sft bat les réponses naïves pour `mcqu` et `mcqm` |

- La courbe `eval_mean_token_accuracy` mélange toutes les évaluations (TRL ignore le préfixe) : utiliser `*_token_accuracy`.
- Les mesures portent sur les lettres : une réponse sans lettre (format non appris) compte 0.
- `*_frequente`, `*_tout_cocher` : réponses naïves, identiques pour toutes les questions (réponse la plus fréquente du train, A à E). Elles ne dépendent pas du modèle et donnent le niveau à dépasser.
- La latence (TTFT, TPOT, débit) sera mesurée en phase 3, sur le modèle final servi par vLLM.

## 7. Alignement DPO

Le DPO part du **modèle SFT** et lui apprend à préférer la réponse `chosen` à la réponse `rejected` de chaque paire.

- **Modèle de départ** : modèle de base + adaptateur SFT **fusionné** (les poids LoRA sont ajoutés aux poids du modèle). TRL calcule les log-probabilités directement depuis `lm_head`, ce qui impose de fusionner l'adaptateur SFT avant d'en ajouter un nouveau.
- **Nouvel adaptateur LoRA** entraîné par DPO. La **référence** du DPO est le modèle SFT lui-même : TRL l'obtient en désactivant l'adaptateur, sans charger un second modèle en mémoire.
- **Données** (`DPO_DATA_VERSION`, v5 depuis l'essai B ; le SFT reste en v4), orientées triage et distinctes de celles du SFT, mélangées pour l'entraînement, évaluées séparément par langue :
  - **français** (MediQAl `mcqu` et `mcqm`) : `chosen` = corrigé, `rejected` = corrigé avec une erreur construite (omission, ajout ou substitution d'une proposition), sur des cas cliniques non retenus pour le SFT ;
  - **anglais** (UltraMedical) : préférences vérifiées par le corrigé (QCM) ou par un écart de note net (ChatDoctor), équilibrées sur la longueur ; test = paires revues par des experts (Medical RewardBench).
- **Reprise après redémarrage** : exécuter d'abord la section 1 (configuration), puis cette section. Le tokenizer et l'adaptateur SFT sont relus depuis `SFT_ADAPTER_DIR`. Avant la validation (7.4), exécuter aussi la cellule des fonctions de génération (section 6).
- **Relancer seulement le DPO** (nouvel essai, même SFT) : section 1, la cellule des fonctions de génération (section 6), puis toute la section 7.
- **Relancer seulement la validation (7.4)**, sans réentraîner : section 1, puis les deux premières cellules de la section 7 (hyperparamètres, données 7.1), la cellule des fonctions de génération (section 6), et enfin la section 7.4. Sa première cellule recharge l'adaptateur DPO sauvegardé.

In [3]:
# Paires DPO de la phase 1 (anonymisées), par langue. Le SFT reste en v4 (SFT_DATA_DIR).
# v5 (essai B) : fr, deux fois plus de mcqm et une majorité de substitutions ; en, moins de paires et plus courtes
DPO_DATA_VERSION = "v5"
DPO_DATA_DIR = Path("../data") / DPO_DATA_VERSION / "anonymized"
DPO_DATASETS = {"fr": "dpo_mediqal", "en": "dpo_ultramedical"}
DPO_COLUMNS = ["source", "lang", "pair_check", "triage", "prompt", "chosen", "rejected"]  # colonnes communes aux jeux

# Hyperparamètres DPO
DPO_BETA = 0.1              # écart toléré avec la référence : plus beta est grand, plus le modèle reste proche du SFT
DPO_LEARNING_RATE = 5e-5    # LoRA : ~1 % des poids entraînés ; à 5e-6 (valeur usuelle en DPO complet), les marges restaient quasi nulles
DPO_MAX_LENGTH = 1024       # prompt + réponse, en tokens
# Perte DPO. Essai A : DPO standard (« sigmoid ») + perte d'imitation sur la chosen (« sft », méthode RPO).
# Le terme « sft » ancre le format court des réponses : en v4, le DPO seul ajoutait des explications dans 24 % des
# réponses mcqm (style des réponses anglaises). ["sigmoid"] et [1.0] : DPO standard (run v4 de référence).
DPO_LOSS_TYPE = ["sigmoid", "sft"]
DPO_LOSS_WEIGHTS = [1.0, 1.0]
DPO_N_EVAL = 20 if SMALL_RUN else None  # questions par source pour la génération (7.4) ; None : toutes
DPO_BATCH_SIZE = 16                    # génération par lots : à baisser en cas d'OutOfMemoryError

SFT_ADAPTER_DIR = Path("../outputs") / RUN_NAME / "final"  # adaptateur SFT de la section 5
# Perte dans le nom du run, sauf DPO standard (nom des runs précédents inchangé) : par exemple « -sigmoid+sft »
DPO_LOSS_TAG = "" if DPO_LOSS_TYPE == ["sigmoid"] else "-" + "+".join(
    loss if weight == 1 else f"{loss}{weight:g}" for loss, weight in zip(DPO_LOSS_TYPE, DPO_LOSS_WEIGHTS))
DPO_RUN_NAME = (f"dpo-lora-r16-{DPO_DATA_VERSION}{DPO_LOSS_TAG}-beta{DPO_BETA}-lr{DPO_LEARNING_RATE}-seed{SEED}"
                + ("-small" if SMALL_RUN else ""))
DPO_OUTPUT_DIR = Path("../outputs") / DPO_RUN_NAME  # checkpoints, adaptateur DPO et modèle final fusionné
print(f"SFT : {SFT_ADAPTER_DIR}\nDPO : {DPO_OUTPUT_DIR}")

SFT : ../outputs/sft-lora-r16-v4-seed42/final
DPO : ../outputs/dpo-lora-r16-v5-sigmoid+sft-beta0.1-lr5e-05-seed42


### 7.1 Données DPO

In [4]:
# Tokenizer sauvegardé avec l'adaptateur SFT : même gabarit qu'à l'entraînement SFT
tokenizer = AutoTokenizer.from_pretrained(SFT_ADAPTER_DIR)


def pair_length(example):
    # Longueur en tokens de la plus longue des deux conversations (prompt + chosen, prompt + rejected)
    return max(
        len(tokenizer(tokenizer.apply_chat_template(example["prompt"] + answer, tokenize=False), add_special_tokens=False)["input_ids"])
        for answer in (example["chosen"], example["rejected"])
    )


DPO_SMALL_SIZES = {"train": 8, "validation": 2, "test": 4}  # petit run : paires par groupe
DPO_SMALL_LONGEST = 4                                       # + paires les plus longues du train, par langue

dpo_by_lang = {}
for lang, stem in DPO_DATASETS.items():
    files = {split: str(DPO_DATA_DIR / f"{stem}_{split}.anonymized.jsonl") for split in ("train", "validation", "test")}
    ds = load_dataset("json", data_files=files)
    # Paires trop longues : TRL couperait la fin de la réponse, on les écarte
    ds = ds.filter(lambda example: pair_length(example) <= DPO_MAX_LENGTH)
    if SMALL_RUN:
        # Quelques paires de chaque groupe évalué en 7.4 (fr : source × type d'erreur ; en : triage × chosen plus longue
        # ou plus courte), plus les paires les plus longues du train : mémoire GPU testée dans le pire cas
        for split, size in DPO_SMALL_SIZES.items():
            d = ds[split]
            groups = (list(zip(d["source"], d["rejected_type"])) if lang == "fr" else
                      [(t, len(c[0]["content"]) > len(r[0]["content"])) for t, c, r in zip(d["triage"], d["chosen"], d["rejected"])])
            keep = [i for group in sorted(set(groups)) for i in [j for j, g in enumerate(groups) if g == group][:size]]
            if split == "train":
                length = [pair_length(example) for example in d]
                keep += sorted(range(len(d)), key=lambda i: length[i], reverse=True)[:DPO_SMALL_LONGEST]
            ds[split] = d.select(sorted(set(keep)))
    if lang == "fr":  # type d'erreur de la rejected (omission, ajout, substitution), pour la validation 7.4
        fr_rejected_types = {split: d["rejected_type"] for split, d in ds.items()}
    dpo_by_lang[lang] = ds.select_columns(DPO_COLUMNS)
    print(f"{lang}: " + ", ".join(f"{split} {len(d)}" for split, d in ds.items()))

# Entraînement et validation : les langues mélangées
dpo_train = concatenate_datasets([ds["train"] for ds in dpo_by_lang.values()]).shuffle(seed=SEED)
dpo_validation = concatenate_datasets([ds["validation"] for ds in dpo_by_lang.values()]).shuffle(seed=SEED)
print(f"DPO : train {len(dpo_train)}, validation {len(dpo_validation)}")

fr: train 1507, validation 201, test 448
en: train 727, validation 73, test 750
DPO : train 2234, validation 274


In [5]:
# Une paire telle que la verra le modèle
example = dpo_train[0]
print(tokenizer.apply_chat_template(example["prompt"], add_generation_prompt=True, tokenize=False))
print(f"--- CHOSEN :\n{example['chosen'][0]['content']}\n--- REJECTED :\n{example['rejected'][0]['content']}")

<|im_start|>system
You are an expert medical assistant. Answer accurately and clearly.<|im_end|>
<|im_start|>user
Two days after undergoing emergency cardiac catherization for myocardial infarction, a 68-year-old woman has pain in her toes. During the intervention, she was found to have an occluded left anterior descending artery and 3 stents were placed. She has hypertension, hypercholesterolemia, and coronary artery disease. Prior to admission, her medications were metoprolol, enalapril, atorvastatin, and aspirin. Her temperature is 37.3°C (99.1°F), pulse is 93/min, and blood pressure is 115/78 mm Hg. Examination shows discoloration of the toes of both feet. A photograph of the right foot is shown. The lesions are cool and tender to palpation. The rest of the skin on the feet is warm; femoral and pedal pulses are palpable bilaterally. This patient is at increased risk for which of the following conditions?

A. Acute kidney injury
B. Basophilia
C. Permanent flexion contracture
D. Migr

### 7.2 Modèle de départ : modèle SFT fusionné

In [6]:
# Libère la mémoire GPU du SFT si la section 5 vient de tourner
if "trainer" in globals():
    del trainer, model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

# Modèle de base + adaptateur SFT fusionné : point de départ et référence du DPO
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype)
sft_model = PeftModel.from_pretrained(base_model, str(SFT_ADAPTER_DIR)).merge_and_unload().to(device)
print(f"{sft_model.num_parameters() / 1e9:.2f} B paramètres")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1401.77it/s]


1.72 B paramètres


### 7.3 Entraînement DPO

- **LoRA** : même configuration que pour le SFT, sur un nouvel adaptateur.
- **Perte** (`DPO_LOSS_TYPE`) : DPO standard, plus, dans l'essai A, une perte d'imitation sur la `chosen` (`sft`, méthode RPO). Ce terme garde le modèle proche des réponses préférées, au format court. La perte affichée inclut ce terme : comparer les runs sur les préférences (`rewards/*`), pas sur la perte.
- **Une époque** : le DPO sur-apprend vite. Au-delà, le modèle mémorise les paires sans mieux généraliser.
- **Batch effectif de 16** (2 × 8) : chaque exemple contient deux réponses, donc deux fois plus de tokens qu'en SFT.
- **Learning rate de 5e-5**, avec montée progressive sur 10 % des étapes. Plus élevé qu'en DPO complet (5e-6 à 5e-7) : avec LoRA, seule une petite partie des poids est entraînée. Au premier run, 5e-6 laissait la marge des récompenses quasi nulle sur le français et la courbe n'avait pas atteint son plateau. Si `logps/chosen` chute fortement, revenir à 2e-5.

In [7]:
dpo_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

# Suivi MLflow : même serveur que le SFT (connexion faite en section 5), expérience dédiée au DPO
if MLFLOW_BACKEND == "dagshub":
    import dagshub

    dagshub.init(repo_owner=DAGSHUB_REPO_OWNER, repo_name=DAGSHUB_REPO_NAME, mlflow=True)
else:
    mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("qwen3-1.7b-dpo-lora")
mlflow.start_run(run_name=DPO_RUN_NAME)
mlflow.log_params({
    "model": MODEL_NAME, "data_version": DATA_VERSION, "dpo_data_version": DPO_DATA_VERSION, "sft_adapter": str(SFT_ADAPTER_DIR), "small_run": SMALL_RUN, "device": device,
    # loss_type et loss_weights : enregistrés par le Trainer avec les autres arguments (un paramètre MLflow ne peut
    # pas recevoir deux valeurs différentes : l'envoi des arguments échouerait)
    "beta": DPO_BETA, "lora_r": dpo_lora_config.r, "lora_alpha": dpo_lora_config.lora_alpha,
    **{f"pairs_{lang}_{split}": len(d) for lang, ds in dpo_by_lang.items() for split, d in ds.items()},
})

dpo_args = DPOConfig(
    output_dir=str(DPO_OUTPUT_DIR),
    run_name=DPO_RUN_NAME,
    seed=SEED,
    # DPO
    beta=DPO_BETA,
    loss_type=DPO_LOSS_TYPE,  # DPO standard (+ imitation de la chosen dans l'essai A)
    loss_weights=DPO_LOSS_WEIGHTS,
    max_length=DPO_MAX_LENGTH,
    # Optimisation
    num_train_epochs=1,
    max_steps=10 if SMALL_RUN else -1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,  # batch effectif = 16 paires
    learning_rate=DPO_LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,  # 10 % des étapes en montée progressive du learning rate
    gradient_checkpointing=True,
    bf16=bf16,
    fp16=dtype == torch.float16,
    # Évaluation et checkpoints
    eval_strategy="steps",
    eval_steps=5 if SMALL_RUN else 25,
    save_steps=5 if SMALL_RUN else 25,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    # Logs
    logging_steps=1 if SMALL_RUN else 5,
    report_to="mlflow",
)

dpo_trainer = DPOTrainer(
    model=sft_model,
    args=dpo_args,
    train_dataset=dpo_train,
    eval_dataset=dpo_validation,
    processing_class=tokenizer,
    peft_config=dpo_lora_config,
)
dpo_trainer.model.print_trainable_parameters()

Accessing as ObiYohan

Initialized MLflow to track repo "obiyohan/my-ml-mlflow"

Repository obiyohan/my-ml-mlflow initialized!

Tokenizing train dataset: 100%|██████████| 2234/2234 [00:07<00:00, 304.09 examples/s]
Dropping fully truncated examples from train dataset: 100%|██████████| 2234/2234 [00:00<00:00, 27748.72 examples/s]
Tokenizing eval dataset: 100%|██████████| 274/274 [00:00<00:00, 293.83 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 274/274 [00:00<00:00, 20429.83 examples/s]

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


In [8]:
# Reprise après interruption : dpo_trainer.train(resume_from_checkpoint=True)
dpo_result = dpo_trainer.train()
dpo_result.metrics

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
25,1.090447,1.136549,0.513649,408553.000000,4.829501,4.808793,0.845980,-0.044521,-0.175053,0.670290,0.130532,-54.419316,-53.474309
50,1.090165,1.129574,0.531716,834370.000000,4.414384,4.455297,0.840301,-0.309014,-0.767553,0.710145,0.458539,-57.064239,-59.399307
75,0.951523,1.126388,0.527190,1264054.000000,4.272194,4.335146,0.839907,-0.302488,-0.891128,0.724638,0.588640,-56.998982,-60.635053
100,1.057622,1.130874,0.521390,1683655.000000,4.071910,4.140545,0.839601,-0.329160,-1.026415,0.735507,0.697255,-57.265698,-61.987926
125,1.093248,1.126545,0.528555,2106529.000000,4.094740,4.177819,0.840719,-0.340171,-1.008719,0.731884,0.668548,-57.375808,-61.810967
140,1.115550,1.124445,0.528808,2356164.000000,4.097741,4.180782,0.840870,-0.341565,-1.009853,0.728261,0.668288,-57.389751,-61.822306


{'train_runtime': 1840.9062,
 'train_samples_per_second': 1.214,
 'train_steps_per_second': 0.076,
 'total_flos': 2.462108265639936e+16,
 'train_loss': 1.0521296330860683,
 'epoch': 1.0}

In [9]:
# Sauvegarde de l'adaptateur DPO (meilleur checkpoint) et de la configuration du run
DPO_FINAL_DIR = DPO_OUTPUT_DIR / "final"
dpo_trainer.save_model(str(DPO_FINAL_DIR))
tokenizer.save_pretrained(str(DPO_FINAL_DIR))

dpo_run_config = {
    "model": MODEL_NAME,
    "sft_adapter": str(SFT_ADAPTER_DIR),  # l'adaptateur DPO s'applique au modèle SFT fusionné
    "dpo_data": str(DPO_DATA_DIR),
    "seed": SEED,
    "small_run": SMALL_RUN,
    "beta": DPO_BETA,
    "loss_type": DPO_LOSS_TYPE,
    "loss_weights": DPO_LOSS_WEIGHTS,
    "lora": {"r": dpo_lora_config.r, "alpha": dpo_lora_config.lora_alpha, "dropout": dpo_lora_config.lora_dropout},
    "data": {f"{lang}_{split}": len(d) for lang, ds in dpo_by_lang.items() for split, d in ds.items()},
    "training_args": dpo_args.to_dict(),
}
(DPO_FINAL_DIR / "run_config.json").write_text(json.dumps(dpo_run_config, indent=2, default=str), encoding="utf-8")

5084

### 7.4 Validation DPO

1. **Préférences** sur la validation et le test de chaque langue (test anglais : revu par des experts), et sur trois découpages du test :
   - **anglais, selon la longueur** (`chosen` plus longue ou plus courte que la `rejected`) : une précision très différente entre les deux révèle une préférence apprise pour la longueur plutôt que pour le fond ;
   - **anglais, triage** : paires dont la question décrit un patient, avec constantes ou vocabulaire d'urgence (`triage`), le plus proche du cas d'usage ;
   - **français, selon le type d'erreur de la `rejected`** : *omission* (une bonne proposition retirée), *ajout* (une mauvaise ajoutée), *substitution* (l'une remplacée par l'autre). Un modèle qui a appris « cocher moins » plutôt que « cocher juste » réussit les ajouts et rate les omissions.

   La *récompense implicite* d'une réponse mesure combien le modèle DPO la juge plus probable que le modèle SFT. On en tire :
   - `reward_accuracy` : part des paires où la récompense de la `chosen` dépasse celle de la `rejected` (0,5 = hasard) ;
   - `reward_margin` : écart moyen entre les deux récompenses ;
   - `logps_chosen` / `logps_rejected` : log-probabilités moyennes des deux réponses sous le modèle DPO.
2. **Réponses générées** sur les QCM français (`mcqu` et `mcqm`, chaque question avec l'historique de son cas), modèle SFT vs modèle DPO :
   - `test_sft` : toutes les questions du test SFT, les mêmes qu'en section 6 ;
   - `test_dpo` : les questions des paires de test DPO (autres cas cliniques), avec la `chosen` pour corrigé. Environ deux fois plus de questions au total, pour mieux départager les deux modèles.

   Mesures et réponses naïves de la section 6, format des réponses, questions où un seul des deux modèles a la bonne réponse (test de McNemar), et lettres données selon le nombre attendu (`mcqm`). On mesure l'effet du DPO sur les décisions, au-delà des préférences.


In [10]:
# Modèle DPO pour la validation : celui de 7.3 s'il est encore en mémoire, sinon l'adaptateur sauvegardé.
# Après 7.5, l'adaptateur DPO est fusionné dans les poids : désactiver l'adaptateur ne donnerait plus le modèle SFT
# (référence du DPO) et toutes les récompenses seraient nulles. Il faut alors recharger.
from peft import set_peft_model_state_dict
from safetensors.torch import load_file

DPO_FINAL_DIR = DPO_OUTPUT_DIR / "final"
dpo_in_memory = "dpo_trainer" in globals() and any("lora_" in name for name, _ in dpo_trainer.model.named_parameters())
if not dpo_in_memory:
    for name in ("dpo_trainer", "dpo_model", "merged_model", "sft_model", "base_model", "trainer", "model"):
        globals().pop(name, None)
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    # Comme en 7.2 et 7.3 : modèle SFT fusionné + nouvel adaptateur ; la référence reste le modèle SFT (adaptateur désactivé).
    # Ne pas passer un PeftModel déjà chargé : TRL prendrait une copie de l'adaptateur DPO comme référence.
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype)
    sft_model = PeftModel.from_pretrained(base_model, str(SFT_ADAPTER_DIR)).merge_and_unload().to(device)
    eval_args = DPOConfig(
        output_dir=str(DPO_OUTPUT_DIR / "eval"), seed=SEED, beta=DPO_BETA, loss_type=DPO_LOSS_TYPE, loss_weights=DPO_LOSS_WEIGHTS,
        max_length=DPO_MAX_LENGTH,
        per_device_eval_batch_size=4, bf16=bf16, fp16=dtype == torch.float16, report_to="none",
    )
    dpo_trainer = DPOTrainer(
        model=sft_model, args=eval_args, train_dataset=dpo_validation.select(range(1)),  # train_dataset exigé par TRL, inutilisé
        processing_class=tokenizer, peft_config=LoraConfig.from_pretrained(str(DPO_FINAL_DIR)),
    )
    # Poids de l'adaptateur DPO sauvegardé en 7.3, dans le nouvel adaptateur
    result = set_peft_model_state_dict(dpo_trainer.model, load_file(str(DPO_FINAL_DIR / "adapter_model.safetensors")))
    assert not result.unexpected_keys, result.unexpected_keys[:5]
    assert not [key for key in result.missing_keys if "lora_" in key], "poids LoRA manquants"
print(f"Modèle DPO : {'en mémoire (7.3)' if dpo_in_memory else f'rechargé depuis {DPO_FINAL_DIR}'}")

Modèle DPO : en mémoire (7.3)


In [11]:
# 1. Préférences, par split et par langue
# (TRL nomme ses métriques « eval_rewards/... » quel que soit le préfixe : on les renomme ici)
eval_sets = {f"{split}_{lang}": ds[split] for lang, ds in dpo_by_lang.items() for split in ("validation", "test")}

# Test anglais selon la longueur : la chosen est-elle plus longue ou plus courte que la rejected ?
test_en = dpo_by_lang["en"]["test"]
chosen_longer = [len(c[0]["content"]) > len(r[0]["content"]) for c, r in zip(test_en["chosen"], test_en["rejected"])]
eval_sets["test_en_chosen_longer"] = test_en.select([i for i, longer in enumerate(chosen_longer) if longer])
eval_sets["test_en_chosen_shorter"] = test_en.select([i for i, longer in enumerate(chosen_longer) if not longer])

# Test anglais, paires proches du triage (patient décrit, constantes ou vocabulaire d'urgence)
eval_sets["test_en_triage"] = test_en.select([i for i, triage in enumerate(test_en["triage"]) if triage])

# Test français selon le type d'erreur de la rejected
for rejected_type in ("omission", "ajout", "substitution"):
    indices = [i for i, t in enumerate(fr_rejected_types["test"]) if t == rejected_type]
    eval_sets[f"test_fr_{rejected_type}"] = dpo_by_lang["fr"]["test"].select(indices)

# « loss » inclut le terme d'imitation (essai A) : comparer les runs sur reward_accuracy et reward_margin
dpo_rows = {}
for name, dataset in eval_sets.items():
    if len(dataset) == 0:  # petit run : un sous-ensemble peut être vide
        continue
    m = dpo_trainer.evaluate(dataset, metric_key_prefix=name)
    dpo_rows[name] = {
        "pairs": len(dataset),
        "loss": m[f"{name}_loss"],
        "reward_accuracy": m["eval_rewards/accuracies"],
        "reward_margin": m["eval_rewards/margins"],
        "logps_chosen": m["eval_logps/chosen"],
        "logps_rejected": m["eval_logps/rejected"],
    }
dpo_summary = pd.DataFrame(dpo_rows).T
dpo_summary

Tokenizing eval dataset: 100%|██████████| 201/201 [00:00<00:00, 267.45 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 201/201 [00:00<00:00, 18222.70 examples/s]


Training Loss,Validation Loss,Step,Fr Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1.115550,No log,140,0.759144,0.057834,2356164.000000,3.270993,3.410938,0.963587,-0.054944,-0.139305,0.681373,0.084361,-3.014079,-4.494397


Tokenizing eval dataset: 100%|██████████| 448/448 [00:01<00:00, 282.48 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 448/448 [00:00<00:00, 18613.10 examples/s]


Training Loss,Validation Loss,Step,Fr Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1.115550,No log,140,0.772180,0.064009,2356164.000000,3.318372,3.308348,0.955890,-0.045418,-0.133234,0.678571,0.087816,-3.059869,-4.489579


Tokenizing eval dataset: 100%|██████████| 73/73 [00:00<00:00, 328.10 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 73/73 [00:00<00:00, 8135.19 examples/s]


Training Loss,Validation Loss,Step,En Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1.115550,No log,140,1.286324,0.839118,2356164.000000,4.728028,4.781251,0.750582,-1.143587,-3.386673,0.921053,2.243086,-205.094072,-220.154632


... (11 outputs left out)

Training Loss,Validation Loss,Step,Fr Ajout Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1.115550,No log,140,0.751593,0.058556,2356164.000000,3.100833,3.073575,0.960153,-0.036083,-0.130182,0.712500,0.094099,-3.406631,-5.103068


Tokenizing eval dataset: 100%|██████████| 289/289 [00:01<00:00, 275.22 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 289/289 [00:00<00:00, 20205.26 examples/s]


Training Loss,Validation Loss,Step,Fr Substitution Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1.115550,No log,140,0.780887,0.067843,2356164.000000,3.427530,3.462162,0.949627,-0.041342,-0.142568,0.698630,0.101226,-2.634188,-4.338040


,pairs,loss,reward_accuracy,reward_margin,logps_chosen,logps_rejected
validation_fr,201.0,0.759144,0.681373,0.084361,-3.014079,-4.494397
test_fr,448.0,0.772180,0.678571,0.087816,-3.059869,-4.489579
validation_en,73.0,1.286324,0.921053,2.243086,-205.094072,-220.154632
test_en,750.0,1.391423,0.742021,0.654001,-348.912947,-257.706794
test_en_chosen_longer,524.0,1.444900,0.744275,0.472935,-379.336714,-230.978145
test_en_chosen_shorter,226.0,1.257847,0.719298,1.093948,-279.510597,-320.814030
test_en_triage,156.0,1.400629,0.750000,0.949971,-331.584403,-263.915024
test_fr_omission,79.0,0.774334,0.525000,0.022824,-4.200042,-4.245770
test_fr_ajout,80.0,0.751593,0.712500,0.094099,-3.406631,-5.103068
test_fr_substitution,289.0,0.780887,0.698630,0.101226,-2.634188,-4.338040


In [12]:
# 2. Réponses générées sur les QCM français : modèle SFT (adaptateur DPO désactivé) vs modèle DPO.
#    test_sft : toutes les questions du test SFT, les mêmes qu'en section 6 ;
#    test_dpo : questions des paires de test DPO (autres cas cliniques), corrigé = chosen.
assert "generate_batch" in globals(), "Exécuter d'abord la cellule des fonctions de génération (section 6)"
DPO_GENERATION_SOURCES = {"mcqu": MAX_NEW_TOKENS, "mcqm": 192}  # réponses mcqm : listes de 2 à 5 options
DPO_ANSWER_COLUMNS = ["sft", "dpo", *NAIVE_ANSWERS]

dpo_model = dpo_trainer.model.eval()
dpo_model.config.use_cache = True  # accélère la génération

frames, generation_prompts = [], []
for config, max_new_tokens in DPO_GENERATION_SOURCES.items():
    source = f"sft_mediqal_{config}"
    sft_prompts, sft_references = test_questions(source, DPO_N_EVAL)
    pairs = dpo_by_lang["fr"]["test"].filter(lambda example: example["source"] == f"MediQAl-{config}")
    pair_prompts, pair_references = pairs["prompt"][:DPO_N_EVAL], [chosen[0]["content"] for chosen in pairs["chosen"]][:DPO_N_EVAL]
    prompts, references = sft_prompts + pair_prompts, sft_references + pair_references
    with dpo_model.disable_adapter():
        sft_answers = generate_batch(prompts, DPO_BATCH_SIZE, max_new_tokens, desc=f"SFT {config}", model=dpo_model)
    dpo_answers = generate_batch(prompts, DPO_BATCH_SIZE, max_new_tokens, desc=f"DPO {config}", model=dpo_model)
    train_data = load_dataset("json", data_files={"train": str(SFT_DATA_DIR / f"{source}_train.anonymized.jsonl")})["train"]
    frames.append(pd.DataFrame({
        "source": source,
        "test_set": ["test_sft"] * len(sft_prompts) + ["test_dpo"] * len(pair_prompts),
        "question": [prompt[-1]["content"][-200:] for prompt in prompts],
        "reference": references,
        "sft": sft_answers,
        "dpo": dpo_answers,
        **naive_answers(question_examples(train_data)[1]),  # mêmes réponses pour toutes les questions de la source
    }))
    generation_prompts += prompts
dpo_generations = add_scores(pd.concat(frames, ignore_index=True), DPO_ANSWER_COLUMNS, generation_prompts)
dpo_generations.to_csv(DPO_FINAL_DIR / "test_fr_generations.csv", index=False)

# Questions où un seul des deux modèles a exactement la bonne réponse (test de McNemar : ce sont elles qui comptent)
paired = pd.DataFrame({
    "ensemble": paired_comparison(dpo_generations, "dpo", "sft"),
    **{f"{source} / {test_set}": paired_comparison(group, "dpo", "sft")
       for (source, test_set), group in dpo_generations.groupby(["source", "test_set"])},
}).T
only_dpo, only_sft = int(paired.loc["ensemble", "only_dpo"]), int(paired.loc["ensemble", "only_sft"])
display(paired)

# Mesures par source, puis lettres données selon le nombre attendu (mcqm), puis moyennes sur l'ensemble
display(pd.concat({source: score_table(group, DPO_ANSWER_COLUMNS) for source, group in dpo_generations.groupby("source")}))
display(letters_by_expected(dpo_generations[dpo_generations["source"] == "sft_mediqal_mcqm"], ["sft", "dpo"]))
dpo_comparison = score_table(dpo_generations, DPO_ANSWER_COLUMNS)
dpo_comparison


DPO mcqm: 100%|██████████| 28/28 [06:09<00:00, 13.20s/lot]


,questions,only_dpo,only_sft,p_value
ensemble,828.0,54.0,44.0,0.363354
sft_mediqal_mcqm / test_dpo,299.0,18.0,19.0,1.000000
sft_mediqal_mcqm / test_sft,140.0,9.0,3.0,0.145996
sft_mediqal_mcqu / test_dpo,149.0,13.0,7.0,0.263176
sft_mediqal_mcqu / test_sft,240.0,14.0,15.0,1.000000


sft       dpo  frequente  tout_cocher
sft_mediqal_mcqm accuracy         0.109339  0.120729   0.059226     0.050114
                 option_accuracy  0.626879  0.633257   0.559453     0.550342
                 f1               0.673766  0.647252   0.605369     0.694869
                 extra_letters    0.617312  0.020501   0.248292     2.248292
                 format           0.993166  0.990888        NaN          NaN
sft_mediqal_mcqu accuracy         0.431877  0.444730   0.210797     0.000000
                 option_accuracy  0.772751  0.777892   0.684319     0.200000
                 f1               0.431877  0.444730   0.210797     0.333333
                 extra_letters    0.000000  0.000000   0.000000     4.000000
                 format           0.992288  0.982005        NaN          NaN

,sft,dpo,questions
attendues,,,
2,3.122066,2.535211,213
3,3.541667,2.965278,144
4,3.683333,3.066667,60
5,3.772727,3.000000,22


,sft,dpo,frequente,tout_cocher
accuracy,0.260870,0.272947,0.130435,0.026570
option_accuracy,0.695411,0.701208,0.618116,0.385749
f1,0.560125,0.552106,0.419997,0.525017
extra_letters,0.327295,0.010870,0.131643,3.071256
format,0.992754,0.986715,NaN,NaN


In [13]:
# Quelques exemples à relire
dpo_generations.groupby("source")[["source", "question", "reference", "sft", "dpo"]].head(3)

,source,question,reference,sft,dpo
0,sft_mediqal_mcqu,entation étiologique :\n\nPropositions :\nA. B...,E. Episode diarrhéique,E. Episode diarrhéique,E. Episode diarrhéique
1,sft_mediqal_mcqu,r le diagnostic ?\n\nPropositions :\nA. Hémocu...,D. Sérodiagnostic de Yersinia enterocolitica,E. Examen parasitologique des selles,E. Examen parasitologique des selles
2,sft_mediqal_mcqu,Uvéite\nB. Adénopathies hilobronchiques bilat...,B. Adénopathies hilobronchiques bilatérales,E. Adénopathies épitrochléennes bilatérales,E. Adénopathies épitrochléennes bilatérales
389,sft_mediqal_mcqm,. Myélogramme\nC. Gastroscopie avec biopsie\nD...,- B. Myélogramme\n- D. Dosages de vitamine B12...,- A. Tubage gastrique\n- B. Myélogramme\n- C. ...,- C. Gastroscopie avec biopsie\n- D. Dosages d...
390,sft_mediqal_mcqm,B. Trouble d'absorption de la vitamine B12\nC....,- C. Carence d'apport en folates\n- D. Trouble...,- A. Carence d'apport en vitamine B12\n- C. Ca...,- B. Trouble d'absorption de la vitamine B12\n...
391,sft_mediqal_mcqm,phénotypes rhésus et Kell\nC. Recherche et id...,- B. Détermination des phénotypes rhésus et Ke...,- A. Recherche d'anticorps anti-HLA\n- B. Déte...,- C. Recherche et identification des anticorps...


In [14]:
# Métriques finales dans MLflow, puis clôture du run
# Validation relancée seule : les métriques rejoignent le run DPO existant (même nom), au lieu d'en créer un nouveau
if mlflow.active_run() is None:
    if MLFLOW_BACKEND == "dagshub":
        import dagshub

        dagshub.init(repo_owner=DAGSHUB_REPO_OWNER, repo_name=DAGSHUB_REPO_NAME, mlflow=True)
    else:
        mlflow.set_tracking_uri("sqlite:///../mlflow.db")
    previous = mlflow.search_runs(
        experiment_names=["qwen3-1.7b-dpo-lora"], filter_string=f"tags.mlflow.runName = '{DPO_RUN_NAME}'",
        order_by=["start_time DESC"], max_results=1,
    )
    assert len(previous), f"run {DPO_RUN_NAME} introuvable dans MLflow"
    mlflow.start_run(run_id=previous["run_id"].iloc[0])
mlflow.log_metrics({
    f"{name}_{metric}": value
    for name, row in dpo_summary.iterrows()
    for metric, value in row.items()
})
mlflow.log_metrics({
    f"test_fr_{metric}_{model}": value
    for metric, row in dpo_comparison.iterrows()
    for model, value in row.items()
    if pd.notna(value)  # format : modèles seulement
})
mlflow.log_metrics({"test_fr_only_dpo_correct": only_dpo, "test_fr_only_sft_correct": only_sft,
                    "test_fr_questions": int(paired.loc["ensemble", "questions"]),
                    "test_fr_mcnemar_p": float(paired.loc["ensemble", "p_value"])})
mlflow.end_run()

🏃 View run dpo-lora-r16-v5-sigmoid+sft-beta0.1-lr5e-05-seed42 at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/3/runs/f9df442d14c54d43b39b6fa53363c958
🧪 View experiment at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/3


### 7.5 Modèle final fusionné

Modèle de base + SFT + DPO en un seul jeu de poids complets (environ 3,4 Go en bf16), avec le tokenizer et son gabarit : c'est le modèle à servir avec vLLM en phase 3.

In [15]:
MERGED_DIR = DPO_OUTPUT_DIR / "merged"
merged_model = dpo_trainer.model.merge_and_unload()
merged_model.save_pretrained(str(MERGED_DIR))
tokenizer.save_pretrained(str(MERGED_DIR))
print(f"Modèle final : {MERGED_DIR}")

Writing model shards: 100%|██████████| 1/1 [00:10<00:00, 10.12s/it]

Modèle final : ../outputs/dpo-lora-r16-v5-sigmoid+sft-beta0.1-lr5e-05-seed42/merged


## Lecture des résultats DPO

| Métrique MLflow | Définition | Attendu |
|---|---|---|
| `loss`, `eval_loss` | perte DPO d'entraînement et de validation (0,69 = aucune préférence apprise) | baissent sous 0,69 ; si `eval_loss` remonte : surapprentissage |
| `rewards/accuracies` | part des paires où la `chosen` l'emporte sur la `rejected` | monte nettement au-dessus de 0,5 |
| `rewards/margins` | écart moyen de récompense entre `chosen` et `rejected` | augmente, sans exploser |
| `logps/chosen`, `logps/rejected` | log-probabilités moyennes des deux réponses | `rejected` baisse plus que `chosen` ; si `chosen` chute fortement, baisser le learning rate ou augmenter beta |
| `{validation,test}_{fr,en}_*` | les mêmes métriques, par split et par langue (test anglais : revu par des experts) | proches entre validation et test |
| `test_en_chosen_{longer,shorter}_*` | test anglais, selon que la `chosen` est la plus longue ou la plus courte | précisions proches ; un grand écart = préférence pour la longueur |
| `test_en_triage_*` | test anglais, paires proches du triage (patient décrit, constantes ou urgence) | au moins aussi bon que le test entier |
| `test_fr_{omission,ajout,substitution}_*` | test français, selon le type d'erreur de la `rejected` | précisions proches ; ajout ≫ omission = le modèle a appris à cocher moins |
| `test_fr_{accuracy,option_accuracy,f1,extra_letters}_*` | lettres des réponses générées sur les QCM français (test SFT et questions des paires de test DPO), comparées au corrigé (mêmes mesures et réponses naïves qu'en section 6) | dpo ≥ sft, au-dessus de `frequente` et `tout_cocher` |
| `test_fr_format_*` | part des réponses au format appris (propositions recopiées, sans texte en plus) | dpo ≥ 0,98 |
| `test_fr_only_{dpo,sft}_correct`, `test_fr_mcnemar_p` | questions où un seul des deux modèles a exactement la bonne réponse, et p-value du test de McNemar | only_dpo ≥ only_sft ; p < 0,05 pour un gain significatif |

- Les courbes `eval_rewards/*` mélangent toutes les évaluations (TRL ignore le préfixe) : pour la comparaison finale, utiliser `{validation,test}_{fr,en}_*`.
- **Itérer** : un run par jeu d'hyperparamètres (`DPO_LOSS_TYPE`, `DPO_LOSS_WEIGHTS`, `DPO_BETA`, `DPO_LEARNING_RATE`), avec le même `SEED`. Le nom du run les contient.
- **Comparer les essais** sur les préférences (`reward_accuracy`), les réponses générées et le format, pas sur la perte : avec le terme `sft`, elle ne se compare plus au DPO standard.